# Group 4 - Assignment 3: Airline Delay Prediction

Train and evaluate the predictive model in Python, export the results, and use Tableau for visualization, storytelling, and prescriptive analysis.

## Workflow
1. Load the three source sheets.
2. Integrate them using the assignment relationships.
3. Audit missing values and table grain.
4. Train two regression models.
5. Validate both models on a held-out group of flights.
6. Refit the selected model on all valid records.
7. Export Tableau-ready predictions, metrics, coefficients, and scenarios.

**Model grain:** one row per `Flight_Details` record.  
**Validation:** 80/20 grouped split by `Flight_ID`, so records from the same flight cannot appear in both training and testing data.


## 1. Import libraries and define paths

In [1]:
from itertools import product
from pathlib import Path
from typing import Any, Sequence

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    accuracy_score,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

DATA_PATH = Path("A3_Airline_Dataset.xlsx")
OUTPUT_DIR = Path("a3_model_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

if not DATA_PATH.exists():
    raise FileNotFoundError(
        "Place A3_Airline_Dataset.xlsx in the same folder as this notebook."
    )


def read_sheet(sheet_name: str) -> pd.DataFrame:
    """Read one Excel sheet and narrow pandas' broad return type."""
    result: Any = pd.read_excel(DATA_PATH, sheet_name=sheet_name)

    if not isinstance(result, pd.DataFrame):
        raise TypeError(f"Expected a DataFrame for sheet: {sheet_name}")

    return result


def column(frame: pd.DataFrame, name: str) -> pd.Series:
    """Return one column as a Series with a runtime type check."""
    result: Any = frame.loc[:, name]

    if not isinstance(result, pd.Series):
        raise TypeError(f"Expected one Series for column: {name}")

    return result


def columns(
    frame: pd.DataFrame,
    names: Sequence[str],
) -> pd.DataFrame:
    """Return selected columns as a DataFrame."""
    result: Any = frame.loc[:, list(names)]

    if not isinstance(result, pd.DataFrame):
        raise TypeError(
            "Expected a DataFrame after selecting columns."
        )

    return result

def as_float_array(values: Any) -> np.ndarray:
    """Convert values to a one-dimensional float64 NumPy array."""
    return np.asarray(
        values,
        dtype=np.float64,
    ).reshape(-1)


def scalar_float(value: Any) -> float:
    """Convert a scalar or one-item array to a Python float."""
    return as_float_array(value)[0].item()


def delay_category(values: Any) -> list[str]:
    """Convert numeric delays to Low, Medium, or High categories."""
    numeric_values = as_float_array(values)

    return [
        (
            "Low"
            if value <= 15.0
            else "Medium"
            if value <= 60.0
            else "High"
        )
        for value in numeric_values
    ]

    result = np.empty(
        numeric_values.shape,
        dtype="<U6",
    )

    result[numeric_values <= 15.0] = "Low"

    result[
        (numeric_values > 15.0)
        & (numeric_values <= 60.0)
    ] = "Medium"

    result[numeric_values > 60.0] = "High"

    return cast(NDArray[np.str_], result)

    return np.where(
        numeric_values <= 15.0,
        "Low",
        np.where(
            numeric_values <= 60.0,
            "Medium",
            "High",
        ),
    )


print(f"Dataset: {DATA_PATH.resolve()}")
print(f"Outputs: {OUTPUT_DIR.resolve()}")

Dataset: C:\Python Projects\Advanced_Data_Visualization\A3_Airline_Dataset.xlsx
Outputs: C:\Python Projects\Advanced_Data_Visualization\a3_model_outputs


## 2. Load the data

In [2]:
flights: pd.DataFrame = read_sheet("Flights")
details: pd.DataFrame = read_sheet("Flight_Details")
airports: pd.DataFrame = read_sheet("Airports")

print("Flights:", flights.shape)
print("Flight_Details:", details.shape)
print("Airports:", airports.shape)

Flights: (700, 7)
Flight_Details: (1500, 6)
Airports: (8, 4)


## 3. Audit the data grain

`Flights` contains one row per flight.

`Flight_Details` contains multiple operational records for many flights.


In [3]:
flight_ids: pd.Series = column(flights, "Flight_ID")
detail_flight_ids: pd.Series = column(
    details,
    "Flight_ID",
)
delay_values: pd.Series = column(
    details,
    "Delay_Minutes",
)

detail_counts = details.groupby("Flight_ID").size()

weather_counts = (
    details.groupby("Flight_ID")["Weather"].nunique()
)

traffic_counts = (
    details.groupby("Flight_ID")["Traffic_Level"].nunique()
)

audit: pd.DataFrame = pd.DataFrame(
    {
        "Metric": [
            "Flights rows",
            "Unique Flight_ID in Flights",
            "Flight_Details rows",
            "Flights represented in Flight_Details",
            "Flights without detail records",
            "Missing Delay_Minutes",
            "Maximum detail rows for one flight",
            "Flights with multiple weather values",
            "Flights with multiple traffic values",
        ],
        "Value": [
            int(flights.shape[0]),
            int(flight_ids.nunique()),
            int(details.shape[0]),
            int(detail_flight_ids.nunique()),
            int(
                flight_ids.nunique()
                - detail_flight_ids.nunique()
            ),
            int(delay_values.isna().sum()),
            int(detail_counts.max()),
            int((weather_counts > 1).sum()),
            int((traffic_counts > 1).sum()),
        ],
    }
)

audit

,Metric,Value
0,Flights rows,700
1,Unique Flight_ID in Flights,700
2,Flight_Details rows,1500
3,Flights represented in Flight_Details,621
4,Flights without detail records,79
5,Missing Delay_Minutes,29
6,Maximum detail rows for one flight,10
7,Flights with multiple weather values,374
8,Flights with multiple traffic values,356


## 4. JOIN tables
`Flights ↔ Flight_Details` using `Flight_ID`
`Flights ↔ Airports` using `Origin_City = City`

A unique `Detail_ID` is added so each prediction can be traced to one operational record.


In [4]:
details = details.copy()

details.insert(
    0,
    "Detail_ID",
    [
        f"D{i:04d}"
        for i in range(1, details.shape[0] + 1)
    ],
)

integrated: pd.DataFrame = details.merge(
    flights,
    on="Flight_ID",
    how="left",
    validate="many_to_one",
)

integrated = integrated.merge(
    airports,
    left_on="Origin_City",
    right_on="City",
    how="left",
    validate="many_to_one",
)

df: pd.DataFrame = integrated

flight_dates: pd.Series = column(
    df,
    "Flight_Date",
)

airport_codes: pd.Series = column(
    df,
    "Airport_Code",
)

print("Integrated rows:", df.shape[0])
print(
    "Missing flight matches:",
    int(flight_dates.isna().sum()),
)
print(
    "Missing airport matches:",
    int(airport_codes.isna().sum()),
)

Integrated rows: 1500
Missing flight matches: 0
Missing airport matches: 0


## 5. Prepare valid model records

Rows with missing `Delay_Minutes` are excluded from model training and validation, but they remain in the exported scored dataset.

Category rules:

- Low: 0-15 minutes
- Medium: more than 15 and up to 60 minutes
- High: more than 60 minutes



In [5]:
all_delay_values = column(
    df,
    "Delay_Minutes",
)

valid_mask = all_delay_values.notna()

valid = df.loc[
    valid_mask,
    :,
].copy()

print("Valid delay records:", valid.shape[0])
print(
    "Excluded missing-delay records:",
    df.shape[0] - valid.shape[0],
)

Valid delay records: 1471
Excluded missing-delay records: 29


## 6. Training/test 80/20 split

The split is grouped by `Flight_ID`. This prevents detail records from the same flight from appearing in both the training and test sets.


In [6]:
# Split unique flights, not individual detail rows.
unique_flight_ids = (
    column(valid, "Flight_ID")
    .dropna()
    .unique()
)

train_flight_ids, test_flight_ids = train_test_split(
    unique_flight_ids,
    test_size=0.20,
    random_state=42,
)

train_mask = column(
    valid,
    "Flight_ID",
).isin(train_flight_ids)

test_mask = column(
    valid,
    "Flight_ID",
).isin(test_flight_ids)

train = valid.loc[
    train_mask,
    :,
].copy()

test = valid.loc[
    test_mask,
    :,
].copy()

overlap = set(
    train_flight_ids
).intersection(test_flight_ids)

assert not overlap, (
    "Data leakage detected: some Flight_ID values "
    "appear in both sets."
)

assert train.shape[0] + test.shape[0] == valid.shape[0], (
    "Some valid records were not assigned."
)

print("Training rows:", train.shape[0])
print("Test rows:", test.shape[0])
print(
    "Training flights:",
    column(train, "Flight_ID").nunique(),
)
print(
    "Test flights:",
    column(test, "Flight_ID").nunique(),
)
print("Overlapping flights:", len(overlap))

Training rows: 1160
Test rows: 311
Training flights: 496
Test flights: 124
Overlapping flights: 0


## 7. Model 1 - Weather and traffic


In [7]:
weather_traffic_encoder = OneHotEncoder(
    categories=[
        ["Clear", "Rain", "Fog", "Snow"],
        ["Low", "Medium", "High"],
    ],
    drop="first",
    handle_unknown="ignore",
    sparse_output=False,
)

model1_preprocessor = ColumnTransformer(
    transformers=[
        (
            "conditions",
            weather_traffic_encoder,
            ["Weather", "Traffic_Level"],
        )
    ],
    remainder="drop",
)

model1 = Pipeline(
    steps=[
        ("preprocessor", model1_preprocessor),
        ("regression", LinearRegression()),
    ]
)

train_target = as_float_array(
    column(train, "Delay_Minutes")
)

test_target = as_float_array(
    column(test, "Delay_Minutes")
)

model1.fit(
    train,
    train_target,
)

model1_test_prediction = as_float_array(
    model1.predict(test)
)

## 8. Model 2 - Add airport and flight variables

In [8]:
model2_categorical = [
    "Weather",
    "Traffic_Level",
    "Avg_Delay_Risk",
    "Airport_Capacity_Level",
    "Airline",
]

model2_numeric = [
    "Distance_km",
    "Scheduled_Departure_Hour",
]

model2_preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(
                drop="first",
                handle_unknown="ignore",
                sparse_output=False,
            ),
            model2_categorical,
        ),
        ("numeric", "passthrough", model2_numeric),
    ],
    remainder="drop",
)

model2 = Pipeline(
    steps=[
        ("preprocessor", model2_preprocessor),
        ("regression", LinearRegression()),
    ]
)

model2.fit(
    train,
    train_target,
)

model2_test_prediction = as_float_array(
    model2.predict(test)
)



## 9. Evaluate both models on the held-out test set

In [9]:
def evaluate_model(
    name: str,
    actual: np.ndarray,
    predicted: np.ndarray,
) -> dict[str, Any]:
    """Calculate out-of-sample performance metrics."""

    actual_category = delay_category(actual)
    predicted_category = delay_category(predicted)

    mse = mean_squared_error(
        actual,
        predicted,
    )

    return {
        "Model": name,
        "Test_R2": scalar_float(
            r2_score(
                actual,
                predicted,
            )
        ),
        "Test_MAE_Minutes": scalar_float(
            mean_absolute_error(
                actual,
                predicted,
            )
        ),
        "Test_RMSE_Minutes": scalar_float(
            np.sqrt(mse)
        ),
        "Test_Category_Accuracy": scalar_float(
            accuracy_score(
                actual_category,
                predicted_category,
            )
        ),
        "Training_Rows": train.shape[0],
        "Test_Rows": test.shape[0],
        "Training_Flights": column(
            train,
            "Flight_ID",
        ).nunique(),
        "Test_Flights": column(
            test,
            "Flight_ID",
        ).nunique(),
    }


metric_rows = [
    evaluate_model(
        "Model 1: Weather + Traffic",
        test_target,
        model1_test_prediction,
    ),
    evaluate_model(
        "Model 2: + Airport and Flight Variables",
        test_target,
        model2_test_prediction,
    ),
]

metrics = pd.DataFrame(metric_rows)

base_r2 = scalar_float(
    column(
        metrics,
        "Test_R2",
    ).iloc[0]
)

base_mae = scalar_float(
    column(
        metrics,
        "Test_MAE_Minutes",
    ).iloc[0]
)

metrics.loc[:, "R2_Gain_vs_Model1"] = (
    column(
        metrics,
        "Test_R2",
    ).astype(float)
    - base_r2
)

metrics.loc[:, "MAE_Change_vs_Model1"] = (
    column(
        metrics,
        "Test_MAE_Minutes",
    ).astype(float)
    - base_mae
)

metrics.round(4)

,Model,Test_R2,Test_MAE_Minutes,Test_RMSE_Minutes,Test_Category_Accuracy,Training_Rows,Test_Rows,Training_Flights,Test_Flights,R2_Gain_vs_Model1,MAE_Change_vs_Model1
0,Model 1: Weather + Traffic,0.7064,14.0263,17.9484,0.7524,1160,311,496,124,0.0000,0.0000
1,Model 2: + Airport and Flight Variables,0.7083,14.0412,17.8895,0.7524,1160,311,496,124,0.0019,0.0149


## 10. Select and refit the deployment model

Model 1 is selected when Model 2 provides negligible improvement. The selected model is then refitted on all valid-delay records before scoring the complete dataset.


In [10]:
# Model 1 is selected as the deployment model because
# Model 2 provides negligible improvement.


# ---------------------------------------------------------
# Store true held-out validation predictions
# ---------------------------------------------------------

validation_predicted_delay = (
    as_float_array(model1_test_prediction)
    .round(1)
)

validation_actual_delay = as_float_array(
    test_target
)

validation_predicted_category = delay_category(
    validation_predicted_delay
)

validation_actual_category = delay_category(
    validation_actual_delay
)

validation_error = (
    validation_actual_delay
    - validation_predicted_delay
).round(1)

validation_abs_error = (
    np.abs(validation_error)
    .round(1)
)

validation_correct = [
    int(actual == predicted)
    for actual, predicted in zip(
        validation_actual_category,
        validation_predicted_category,
    )
]


# Create empty validation columns for all records.
# Only held-out test records will receive values.

df.loc[
    :,
    "Validation_Predicted_Delay",
] = np.nan

df.loc[
    :,
    "Validation_Predicted_Category",
] = pd.NA

df.loc[
    :,
    "Validation_Error",
] = np.nan

df.loc[
    :,
    "Validation_Abs_Error",
] = np.nan

df.loc[
    :,
    "Validation_Correct",
] = pd.NA


# test keeps the original df row indexes,
# so predictions can be assigned back accurately.

df.loc[
    test.index,
    "Validation_Predicted_Delay",
] = validation_predicted_delay

df.loc[
    test.index,
    "Validation_Predicted_Category",
] = validation_predicted_category

df.loc[
    test.index,
    "Validation_Error",
] = validation_error

df.loc[
    test.index,
    "Validation_Abs_Error",
] = validation_abs_error

df.loc[
    test.index,
    "Validation_Correct",
] = validation_correct




deployment_model = Pipeline(
    steps=[
        (
            "preprocessor",
            model1_preprocessor,
        ),
        (
            "regression",
            LinearRegression(),
        ),
    ]
)

# Refit Model 1 using all records with valid delays.
valid_target = as_float_array(
    column(
        valid,
        "Delay_Minutes",
    )
)

deployment_model.fit(
    valid,
    valid_target,
)

# Score all detail records, including rows with missing actual delays.
all_predictions = as_float_array(
    deployment_model.predict(df)
).round(1)

df.loc[
    :,
    "Predicted_Delay",
] = all_predictions

df.loc[
    :,
    "Predicted_Category",
] = delay_category(
    all_predictions
)

# Create the actual delay category only where Delay_Minutes exists.
df.loc[
    :,
    "Actual_Category",
] = pd.NA

actual_mask = column(
    df,
    "Delay_Minutes",
).notna()

actual_delays = as_float_array(
    column(
        df.loc[
            actual_mask,
            :,
        ],
        "Delay_Minutes",
    )
)

df.loc[
    actual_mask,
    "Actual_Category",
] = delay_category(
    actual_delays
)

# Calculate record-level model errors.
delay_numeric = column(
    df,
    "Delay_Minutes",
).astype(float)

prediction_numeric = column(
    df,
    "Predicted_Delay",
).astype(float)

df.loc[
    :,
    "Prediction_Error",
] = (
    delay_numeric
    - prediction_numeric
).round(1)

df.loc[
    :,
    "Abs_Error",
] = column(
    df,
    "Prediction_Error",
).abs().round(1)

# Compare actual and predicted categories.
df.loc[
    :,
    "Prediction_Correct",
] = pd.NA

actual_categories = column(
    df.loc[
        actual_mask,
        :,
    ],
    "Actual_Category",
).astype(str).tolist()

predicted_categories = column(
    df.loc[
        actual_mask,
        :,
    ],
    "Predicted_Category",
).astype(str).tolist()

correct_values = [
    int(actual == predicted)
    for actual, predicted in zip(
        actual_categories,
        predicted_categories,
    )
]

df.loc[
    actual_mask,
    "Prediction_Correct",
] = correct_values

# Preserve the original train/test assignment for Tableau validation.
train_ids = set(
    column(
        train,
        "Detail_ID",
    ).astype(str).tolist()
)

test_ids = set(
    column(
        test,
        "Detail_ID",
    ).astype(str).tolist()
)

df.loc[
    :,
    "Data_Split",
] = "Missing Delay"

detail_id_values = column(
    df,
    "Detail_ID",
).astype(str)

df.loc[
    detail_id_values.isin(train_ids),
    "Data_Split",
] = "Train"

df.loc[
    detail_id_values.isin(test_ids),
    "Data_Split",
] = "Test"

columns(
    df,
    [
        "Detail_ID",
        "Flight_ID",
        "Weather",
        "Traffic_Level",
        "Delay_Minutes",
        "Data_Split",

        # Held-out test predictions
        "Validation_Predicted_Delay",
        "Validation_Predicted_Category",
        "Validation_Error",
        "Validation_Abs_Error",
        "Validation_Correct",

        # Final deployed-model predictions
        "Predicted_Delay",
        "Predicted_Category",
        "Prediction_Error",
        "Abs_Error",
        "Prediction_Correct",
    ],
).head()

,Detail_ID,Flight_ID,Weather,Traffic_Level,Delay_Minutes,Data_Split,Validation_Predicted_Delay,Validation_Predicted_Category,Validation_Error,Validation_Abs_Error,Validation_Correct,Predicted_Delay,Predicted_Category,Prediction_Error,Abs_Error,Prediction_Correct
0,D0001,5122,Rain,High,96.0,Train,NaN,<NA>,NaN,NaN,<NA>,72.5,High,23.5,23.5,1
1,D0002,5025,Rain,Medium,27.0,Train,NaN,<NA>,NaN,NaN,<NA>,47.4,Medium,-20.4,20.4,1
2,D0003,5677,Snow,Low,84.0,Test,77.9,High,6.1,6.1,1,77.8,High,6.2,6.2,1
3,D0004,5269,Rain,Medium,31.0,Train,NaN,<NA>,NaN,NaN,<NA>,47.4,Medium,-16.4,16.4,1
4,D0005,5299,Clear,Medium,17.0,Train,NaN,<NA>,NaN,NaN,<NA>,24.5,Medium,-7.5,7.5,1


## 11. Extract final coefficients for documentation

In [11]:
# Retrieve the fitted preprocessing and regression steps.
preprocessor = deployment_model.named_steps[
    "preprocessor"
]

regression = deployment_model.named_steps[
    "regression"
]

# Confirm that the retrieved objects have the expected types.
if not isinstance(
    preprocessor,
    ColumnTransformer,
):
    raise TypeError(
        "Expected a ColumnTransformer."
    )

if not isinstance(
    regression,
    LinearRegression,
):
    raise TypeError(
        "Expected a LinearRegression model."
    )

# Retrieve and clean the encoded feature names.
raw_feature_names = (
    preprocessor.get_feature_names_out()
)

feature_names = [
    str(name)
    .replace(
        "conditions__",
        "",
    )
    .replace(
        "Weather_",
        "Weather: ",
    )
    .replace(
        "Traffic_Level_",
        "Traffic: ",
    )
    for name in raw_feature_names.tolist()
]

# Retrieve coefficients and intercept.
coefficient_values = as_float_array(
    regression.coef_
)

intercept = scalar_float(
    regression.intercept_
)

coefficient_terms = [
    "Intercept: Clear + Low",
    *feature_names,
]

coefficient_effects = [
    intercept,
    *coefficient_values.tolist(),
]

coefficients = pd.DataFrame(
    {
        "Term": coefficient_terms,
        "Effect_Minutes": coefficient_effects,
    }
)

coefficients.round(2)

,Term,Effect_Minutes
0,Intercept: Clear + Low,13.61
1,Weather: Rain,22.98
2,Weather: Fog,39.37
3,Weather: Snow,64.21
4,Traffic: Medium,10.85
5,Traffic: High,35.87


## 12. Create the 12-condition scenario table

This table supports a Tableau parameter-driven simulator without requiring TabPy or hardcoded model coefficients.


In [12]:
scenario_rows: list[
    tuple[str, str]
] = list(
    product(
        [
            "Clear",
            "Rain",
            "Fog",
            "Snow",
        ],
        [
            "Low",
            "Medium",
            "High",
        ],
    )
)

scenario_grid: pd.DataFrame = pd.DataFrame(
    scenario_rows,
    columns=[
        "Weather",
        "Traffic_Level",
    ],
)

scenario_predictions = as_float_array(
    deployment_model.predict(scenario_grid)
).round(1)

scenario_grid.loc[
    :,
    "Predicted_Delay",
] = scenario_predictions

scenario_grid.loc[
    :,
    "Predicted_Category",
] = delay_category(
    scenario_predictions
)

recommended_actions: list[str] = [
    (
        "SEVERE — thin schedule, stage de-icing, rebook connections"
        if predicted_delay >= 90.0
        else (
            "ELEVATED — add turnaround buffer, "
            "shift discretionary departures"
            if predicted_delay >= 50.0
            else "NORMAL — standard operations"
        )
    )
    for predicted_delay in scenario_predictions
]

scenario_grid.loc[
    :,
    "Recommended_Action",
] = recommended_actions

scenario_grid

,Weather,Traffic_Level,Predicted_Delay,Predicted_Category,Recommended_Action
0,Clear,Low,13.6,Low,NORMAL — standard operations
1,Clear,Medium,24.5,Medium,NORMAL — standard operations
2,Clear,High,49.5,Medium,NORMAL — standard operations
3,Rain,Low,36.6,Medium,NORMAL — standard operations
4,Rain,Medium,47.4,Medium,NORMAL — standard operations
5,Rain,High,72.5,High,"ELEVATED — add turnaround buffer, shift discre..."
6,Fog,Low,53.0,Medium,"ELEVATED — add turnaround buffer, shift discre..."
7,Fog,Medium,63.8,High,"ELEVATED — add turnaround buffer, shift discre..."
8,Fog,High,88.8,High,"ELEVATED — add turnaround buffer, shift discre..."
9,Snow,Low,77.8,High,"ELEVATED — add turnaround buffer, shift discre..."


## 13. Export Tableau files

In [13]:
scored_columns: list[str] = [
  "Detail_ID",
    "Flight_ID",
    "Weather",
    "Traffic_Level",
    "Ticket_Price",
    "Delay_Minutes",
    "Cancelled",
    "Actual_Category",
    "Data_Split",

    # True held-out test results
    "Validation_Predicted_Delay",
    "Validation_Predicted_Category",
    "Validation_Error",
    "Validation_Abs_Error",
    "Validation_Correct",

    # Final model deployed on all records
    "Predicted_Delay",
    "Predicted_Category",
    "Prediction_Error",
    "Abs_Error",
    "Prediction_Correct",
]

scored_details: pd.DataFrame = columns(
    df,
    scored_columns,
).copy()

scored_details.to_csv(
    OUTPUT_DIR
    / "a3_scored_flight_details.csv",
    index=False,
)

metrics.round(6).to_csv(
    OUTPUT_DIR
    / "a3_model_metrics.csv",
    index=False,
)

coefficients.round(4).to_csv(
    OUTPUT_DIR
    / "a3_model_coefficients.csv",
    index=False,
)

scenario_grid.to_csv(
    OUTPUT_DIR
    / "a3_scenario_predictions.csv",
    index=False,
)

audit.to_csv(
    OUTPUT_DIR
    / "a3_data_audit.csv",
    index=False,
)

print("Exported files:")

for file_path in sorted(
    OUTPUT_DIR.glob("*.csv")
):
    print("-", file_path)

Exported files:
- a3_model_outputs\a3_data_audit.csv
- a3_model_outputs\a3_model_coefficients.csv
- a3_model_outputs\a3_model_metrics.csv
- a3_model_outputs\a3_scenario_predictions.csv
- a3_model_outputs\a3_scored_flight_details.csv


## 14. Interpretation

- Model 1 provides the operational prediction used in Tableau.
- Model 2 tests whether airport and flight variables materially improve prediction.
- The test metrics must be reported as out-of-sample results.
- Tableau uses `a3_scored_flight_details.csv` as the enriched replacement for the original `Flight_Details` table.
- `a3_scenario_predictions.csv` powers the what-if simulator.
- `a3_model_metrics.csv` can power compact KPI tiles.
